### View-Based Learning using Compressive Sensing Induced Augmentation on Music

. 


In [2]:
import pandas as pd
from preprocess import run_decode_audio, run_mel, run_audio_unzip
from representation import run_wave_barlow
from evaluation import run_linear_probe

In [ ]:
# Regenerate .npy files from .mp3 (using ffmpeg) 
run_decode_audio("preprocesss/data/fma_small") 

In [ ]:
# Regenerate mel-spectrograms from audio signal for baseline/visualization
run_mel("preprocess/data/fma_small") 

In [ ]:
# Unzip .npy and mel-spectrogram .pt from zip 
run_audio_unzip("fma_small_decoded.zip", "preprocess/data/fma_small")

### Barlow Twins Training

_Note_: For all three models I made the decision to included the Pop genre in training examples; despite not included them in the linear evaluation. Because the dataset is on the small side, pop samples provide examples for the model to further learn invariance of music on. However, pop samples themselves do not have a strong genre identity, so they pollute linear evaluation in general. Thus we include them in training but not in evaluation of representation quality. 

We have 4 training protocols. The first two use only the backprojection/compression induced augmentation. Both use a DCT sensing operator, but the first weights selected rows biased towards lower-energy (a domain specific advantage for audio/music samples) while the latter is more honest, using random subsampling down to $m$ rows. We use this specific ablation because it the low-energy bias demonstrates a specific choice that can be made in general when a task has domain structure to exploit, but in general the choice doesn't matter up to chosen ratio. What is important is that the selected ratio places the cosine similarity of two augmented views within the band $0.5-0.7$, which just requires a higher ratio for random subsampling than for low energy in this case (~$20\%$ versus ~$50\%$). 

The third test is a grid of traditional augmentations. All of which are mentioned in the report but in general are the go to augmentations for audio signals as dictated by literature. 

The final test is a hybrid of traditional and compressive augmentation. My thought was that compressive augmentation (at least for this specific task; I would love to look into how this holds across tasks/input domains) gives a lot of flexibility of the cosine similarity of views by varying $m/N$, hence we can take a strong base for traditional augmentation and then fine tune with the compressive augmentation.

We will evaluate the quality of the learned representation on the $F1$-macro metric of a logistic regression probe (with pop excluded as a confound). 

In [3]:
# low-energy biased sampling
for ratio in [20]:
    run_wave_barlow(ratio=ratio, exclude_genres=["Pop"])

/home/andrew/Projects/spiky/.venv/lib/python3.12/site-packages/torch/_inductor/lowering.py:1814: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
W0529 11:32:28.148000 639184 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


source=wave_barlow_cs_r20_d256_nopop epoch=1/300 train_loss=156.629983 val_loss=91.621797 on_diag=36.8173 off_diag=1096088.9993 sGLt=0.00
source=wave_barlow_cs_r20_d256_nopop epoch=2/300 train_loss=72.323565 val_loss=66.164630 on_diag=16.6495 off_diag=990302.9250 sGLt=0.00
source=wave_barlow_cs_r20_d256_nopop epoch=3/300 train_loss=52.417578 val_loss=54.450774 on_diag=13.7923 off_diag=813168.7939 sGLt=0.00
source=wave_barlow_cs_r20_d256_nopop epoch=4/300 train_loss=42.668642 val_loss=44.975751 on_diag=12.7453 off_diag=644608.3993 sGLt=0.00 P_k=429.73
source=wave_barlow_cs_r20_d256_nopop epoch=5/300 train_loss=35.018748 val_loss=37.979164 on_diag=10.5530 off_diag=548523.9800 sGLt=0.00 P_k=340.01
source=wave_barlow_cs_r20_d256_nopop epoch=6/300 train_loss=30.829198 val_loss=33.459119 on_diag=8.7764 off_diag=493654.4629 sGLt=0.00 P_k=276.67
source=wave_barlow_cs_r20_d256_nopop epoch=7/300 train_loss=26.518418 val_loss=29.163953 on_diag=7.2852 off_diag=437575.9582 sGLt=0.00 P_k=247.93
sour

In [ ]:
# random subsampling 
for ratio in [60]:
    run_wave_barlow(ratio=ratio, uniform=True, exclude_genres=["Pop"])

source=wave_barlow_cs_uniform_r60_d256_nopop epoch=1/300 train_loss=143.451299 val_loss=84.061045 on_diag=34.4393 off_diag=992434.7982 sGLt=0.00
source=wave_barlow_cs_uniform_r60_d256_nopop epoch=2/300 train_loss=65.934533 val_loss=63.304690 on_diag=17.2391 off_diag=921311.0057 sGLt=0.00
source=wave_barlow_cs_uniform_r60_d256_nopop epoch=3/300 train_loss=50.141048 val_loss=51.910081 on_diag=14.3212 off_diag=751778.2446 sGLt=0.00
source=wave_barlow_cs_uniform_r60_d256_nopop epoch=4/300 train_loss=40.536771 val_loss=43.586031 on_diag=13.1369 off_diag=608982.5279 sGLt=0.00 P_k=393.00
source=wave_barlow_cs_uniform_r60_d256_nopop epoch=5/300 train_loss=33.512995 val_loss=36.752264 on_diag=10.8673 off_diag=517699.7707 sGLt=0.00 P_k=330.21
source=wave_barlow_cs_uniform_r60_d256_nopop epoch=6/300 train_loss=28.910850 val_loss=32.019928 on_diag=8.7354 off_diag=465690.2161 sGLt=0.00 P_k=282.55
source=wave_barlow_cs_uniform_r60_d256_nopop epoch=7/300 train_loss=25.051253 val_loss=27.879962 on_dia

In [ ]:
# traditional waveform augmentation of varying level 
for policy in ["w2"]:
    run_wave_barlow(policy=policy, exclude_genres=["Pop"])

### Hybrid Augmentation

Let $M$ be the number of training samples in the training split. Using the traditional and compressive augmentation manifolds, we can measure a "fair" ratio $k/M$ of the total training split used to train the logistic regression model such that validation set performance is relatively optimal compared to using all $M$ samples. Using this ratio of samples, we can push the $1-k/M$ samples to the test set, then split the test set into 3 separate folds. We can then evaluate all learned manifolds and compare their scores on the test sets across several random seeds. Then using this aggregate test score we can compare cosine similarity (alignment), uniformity of learned embeddings, and effective rank of embeddings to determine how we should fine tune the measurement ratio $m/N$ in our hybrid model given a level of traditional augmentation.

We only consider the low-energy bias method for our sensing operator since performance between low-energy bias and random subsampling match when we fine-tune for close cosine similarity. Hence we use the low-energy bias since it has a higher "capacity" achieving the same performance for a lower measurement ratio.

When applying augmentation, we can let $A_i$ be the traditional augmentation operator to the $i$th view (either the first or second), and we can let $C_i$ be the compressive augmentation operator on the $i$th view. We consider the two hybrid methods: 
- $A_1(X) + C_2(X)$
- $A_1(X) + C_2(A_2(X))$

In the first case we take traditional augmentation as the first view and compare against the compressive augmentation on the second view after a traditional augmentation. In the second case, we apply traditional augmentation to both views, then compressive augmentation applied asymmetrically. 

In [8]:
#  w2 vs cs_r30 (asymmetric: time-domain aug vs frequency bottleneck)
run_wave_barlow(ratio=30, policy="w2", exclude_genres=["Pop"])

# w2 vs w2->cs_r20 (chained: both views get w2, one additionally compressed)
run_wave_barlow(ratio=20, policy="w2", method="hybrid_chain", exclude_genres=["Pop"])

source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=1/300 train_loss=275.313772 val_loss=185.788020 on_diag=78.3426 off_diag=2148909.4243 sGLt=0.00
source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=2/300 train_loss=108.208580 val_loss=162.170235 on_diag=100.6535 off_diag=1230335.0314 sGLt=0.00
source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=3/300 train_loss=81.328363 val_loss=166.186310 on_diag=115.4803 off_diag=1014120.8668 sGLt=2.48
source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=4/300 train_loss=68.630292 val_loss=164.748090 on_diag=120.4818 off_diag=885325.0207 sGLt=1.59 P_k=46.57
source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=5/300 train_loss=61.686511 val_loss=158.259401 on_diag=114.2760 off_diag=879667.3336 sGLt=0.00 P_k=28.95
source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=6/300 train_loss=57.888303 val_loss=157.391198 on_diag=114.4647 off_diag=858529.9793 sGLt=0.00 P_k=27.03
source=wave_barlow_hybrid_w2_r30_d256_nopop epoch=7/300 train_loss=54.235089 val_loss=156.742700

checkpoint source=wave_barlow_hybrid_w2_r30_d256_nopop best_epoch=20 best_val_loss=107.864422 path=/home/andrew/Projects/spiky/representation/checkpoints/wave_barlow_hybrid_w2_r30_d256_nopop_fma_small.pt
extracted source=wave_barlow_hybrid_w2_r30_d256_nopop split=training n=6397
extracted source=wave_barlow_hybrid_w2_r30_d256_nopop split=validation n=800
extracted source=wave_barlow_hybrid_w2_r30_d256_nopop split=test n=800
wrote path=/home/andrew/Projects/spiky/representation/data/wave_barlow_fma_small.parquet total_rows=71973


source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=1/300 train_loss=352.355228 val_loss=206.638401 on_diag=102.6927 off_diag=2078913.7371 sGLt=0.00
source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=2/300 train_loss=141.021697 val_loss=146.134628 on_diag=71.2709 off_diag=1497274.0586 sGLt=0.00
source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=3/300 train_loss=107.288928 val_loss=133.974080 on_diag=67.4873 off_diag=1329735.0829 sGLt=0.00
source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=4/300 train_loss=92.213211 val_loss=147.939985 on_diag=93.4899 off_diag=1089000.9407 sGLt=10.42 P_k=184.35
source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=5/300 train_loss=78.670070 val_loss=144.963135 on_diag=96.8786 off_diag=961690.6593 sGLt=8.20 P_k=69.26
source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=6/300 train_loss=70.884914 val_loss=149.470913 on_diag=103.0015 off_diag=929388.5050 sGLt=11.57 P_k=75.48
source=wave_barlow_hybrid_chain_w2_r20_d256_nopop epoch=7/

checkpoint source=wave_barlow_hybrid_chain_w2_r20_d256_nopop best_epoch=18 best_val_loss=90.274072 path=/home/andrew/Projects/spiky/representation/checkpoints/wave_barlow_hybrid_chain_w2_r20_d256_nopop_fma_small.pt
extracted source=wave_barlow_hybrid_chain_w2_r20_d256_nopop split=training n=6397
extracted source=wave_barlow_hybrid_chain_w2_r20_d256_nopop split=validation n=800
extracted source=wave_barlow_hybrid_chain_w2_r20_d256_nopop split=test n=800
wrote path=/home/andrew/Projects/spiky/representation/data/wave_barlow_fma_small.parquet total_rows=79970


PosixPath('/home/andrew/Projects/spiky/representation/data/wave_barlow_fma_small.parquet')

In [2]:
# --- Linear Evaluation --- 
# Measures Representation Quality by sweeping over 12-value logspace grid for C 
# on validation set; f1-macro test determines quality of representation 

df_main = pd.read_parquet("representation/data/wave_barlow_fma_small.parquet") 
df_ablation = pd.read_parquet("representation/data/ablation_uniform/wave_barlow_fma_small.parquet")
df = pd.concat([df_main, df_ablation], ignore_index=True) 

for method, group in df.groupby("method"): 
    result = run_linear_probe(group, exclude_genres=["Pop"]) 
    print(f"{method} val={result['val_f1']:.4f} test={result['test_f1']:.4f}") 

wave_barlow_abt_w2_d256_nopop val=0.5998 test=0.5116
wave_barlow_abt_w3_d256_nopop val=0.5670 test=0.4611
wave_barlow_abt_w4_d256 val=0.5588 test=0.4882
wave_barlow_abt_w4_d256_nopop val=0.5773 test=0.4715
wave_barlow_cs_r10_d256_nopop val=0.5466 test=0.4460
wave_barlow_cs_r15_d256 val=0.5512 test=0.4213
wave_barlow_cs_r20_d256_nopop val=0.5589 test=0.4585
wave_barlow_cs_r30_d256_nopop val=0.5753 test=0.4480
wave_barlow_cs_r40_d256_nopop val=0.5245 test=0.4086
wave_barlow_cs_r50_d256_nopop val=0.5762 test=0.4230
wave_barlow_cs_r60_d256_nopop val=0.6219 test=0.4322
wave_barlow_cs_uniform_r55_d256_nopop val=0.5917 test=0.4444
wave_barlow_cs_uniform_r65_d256_nopop val=0.5802 test=0.4610
wave_barlow_hybrid_chain_w2_r20_d256_nopop val=0.5369 test=0.4441
wave_barlow_hybrid_w2_r30_d256_nopop val=0.5484 test=0.4449
wave_barlow_hybrid_w4_r10_d256 val=0.5442 test=0.4816
wave_barlow_hybrid_w4_r20_d256 val=0.5744 test=0.4662
